# Chien luoc MACD, RSI (Huan luyen  ['High', 'Low', 'Volume', 'MACD'])
### Dua High, RSI vao de huan luyen

# Load data

In [ ]:
import sys
sys.path.append("../Common")
import CommonBinance

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # Model hồi quy tuyến tính
from sklearn.metrics import mean_squared_error # Đánh giá mô hình tot hay xấu
import talib

def detectSignal(symbol, from_date, to_date, timeframe):

	import pandas as pd
	import plotly.graph_objects as go
	import redis
	import numpy as np
	from datetime import datetime

	# ##############################################Step 1: Lấy dữ liệu##############################################
	print("Bat dau lay du lieu")
	print(datetime.now())
	data = CommonBinance.CommonBinance.loaddataBinance_FromTo_Limit100(symbol, from_date, to_date, timeframe)
	print("Ket thuc lay du lieu")
	print(datetime.now())
	print(data)
	# Tra ve Datetime, Open, High, Low, Close, Volume

	# ##############################################Step 2: Chiến lược##############################################  
	# Tính toán các chỉ báo kỹ thuật
	data['MA_fast'] = talib.SMA(data['Close'], timeperiod=10)
	data['MA_slow'] = talib.SMA(data['Close'], timeperiod=20)
	
	# Thêm MA và ATR vào DataFrame
	data['MA'] = talib.SMA(data['Close'], timeperiod=20)
	data['ATR'] = talib.ATR(data['High'], data['Low'], data['Close'], timeperiod=14)

	# Đảm bảo không có giá trị NaN trong Momentum, RSI
	data['MA'] = data['MA'].bfill().ffill()
	data['ATR'] = data['ATR'].bfill().ffill()

	# Bien doc lap la X la 1 bien AREA
	# Bien phu thuoc la Y la bien Price

	# Tạo features (X) và target (y)
	X = data[['Open', 'High', 'Low', 'Volume', 'MA']]  # Dùng dữ liệu của ngày hôm trước để dự đoán
	y = data['Close']  # Giá đóng cửa mà chúng ta muốn dự đoán
  
	# Tiếp tục với việc chia dữ liệu
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
	X_train = X_train.bfill().ffill()
	y_train = y_train.bfill().ffill()

	# Huấn luyện mô hình
	model = LinearRegression()
	model.fit(X_train, y_train) # Luu model xuong 1 file => Load model tu file

	# Đảm bảo không có giá trị NaN trong X trước khi dự đoán
	X = X.bfill().ffill()

	# Dự đoán giá đóng cửa sử dụng X đã điền giá trị NaN
	data['Predicted_Close'] = model.predict(X)

	# Có thể sử dụng MSE để đánh giá mô hình ổn thì làm bước kế tiếp
	# Tính Mean Squared Error (MSE) giữa giá trị thực tế và giá trị dự đoán trên tập kiểm tra
	mse = mean_squared_error(data['Close'], data['Predicted_Close'])
	print(f'Mean Squared Error (MSE): {mse}')

	# Định nghĩa tín hiệu mua dựa trên điều kiện: Dự đoán giá tăng, Momentum dương và RSI dưới 30
	# Mua: MA Fast > MA Slow VÀ ATR < threshold (thị trường ít biến động)
	# Bán: MA Fast < MA Slow VÀ ATR > threshold (thị trường biến động mạnh)

	data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['MA_fast'] > data['MA_slow']) & (data['ATR'] < 40))
	data['Buy_Signal'] = True
	# Định nghĩa tín hiệu bán dựa trên điều kiện: Dự đoán giá giảm, Momentum âm và RSI trên 30
	data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['MA_fast'] < data['MA_slow']) & (data['ATR'] > 80))

	data.to_csv("OG_CRTO_MACD_RSI 28.08.csv")
	
	# ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
	# Nếu có tín hiệu thì đẩy qua Redis
	# Datetime  Open    High    Low	Close   Volume  SMA short_ema   long_ema    MACD    Signal_Line     Buy_Signal      Sell_Signal
	# Tạo kết nối Redis
	r = redis.Redis(host='localhost', port=6379, db=0)
	# Tạo hash key
	hash_key = 'OG_CRTO_ATR_MA 28.08'
	last_record = data.iloc[-2]
	
	# Tracking time > 10s thi bo qua khong chuyen qua redis
	
	# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
	if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
		for field, value in last_record.to_dict().items():
			# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
			if isinstance(value, pd.Timestamp):
				value = value.isoformat()
			elif isinstance(value, (int, np.uint64)):
				value = str(value)
			r.hset(hash_key, field, value)  
			r.hset(hash_key, 'Symbol', symbol)
			r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
		print(last_record)   
	else:
		print(last_record)
		print('Không có tín hiệu!')

	# ##############################################Step 4: Vẽ biểu đồ##############################################  
	
	# import plotly.graph_objects as go

	# # Đảm bảo rằng data['Datetime'] là dạng datetime nếu chưa phải
	# # data['Datetime'] = pd.to_datetime(data['Datetime'])

	# # Tạo biểu đồ Candlestick cho dữ liệu giá
	# fig = go.Figure(data=[go.Candlestick(x=data['Datetime'],  # Sử dụng cột Datetime thay vì index
	#                                     open=data['Open'],
	#                                     high=data['High'],
	#                                     low=data['Low'],
	#                                     close=data['Close'],
	#                                     name='Candlestick')])

	# # Thêm dòng dự đoán giá đóng cửa từ mô hình hồi quy
	# fig.add_trace(go.Scatter(x=data['Datetime'], y=data['Predicted_Close'], mode='lines', name='Predicted Close'))

	# # Thêm điểm mua
	# fig.add_trace(go.Scatter(x=data[data['Buy_Signal']]['Datetime'],
	#                         y=data[data['Buy_Signal']]['Close'],
	#                         mode='markers',
	#                         marker=dict(color='Green', size=10, symbol='triangle-up'),
	#                         name='Buy Signal'))

	# # Thêm điểm bán
	# fig.add_trace(go.Scatter(x=data[data['Sell_Signal']]['Datetime'],
	#                         y=data[data['Sell_Signal']]['Close'],
	#                         mode='markers',
	#                         marker=dict(color='Red', size=10, symbol='triangle-down'),
	#                         name='Sell Signal'))

	# # Thêm Momentum và RSI vào trục Y phụ
	# fig.add_trace(go.Scatter(x=data['Datetime'], y=data['Momentum'], name='Momentum', yaxis='y2'))
	# fig.add_trace(go.Scatter(x=data['Datetime'], y=data['RSI'], name='RSI', yaxis='y3'))

	# # Tùy chỉnh layout để thêm trục Y phụ và cập nhật tiêu đề trục X
	# fig.update_layout(
	#     title=f'Trading Signals for {symbol} based on Linear Regression, Momentum, and RSI',
	#     xaxis_title='Date',
	#     yaxis_title='Price',
	#     xaxis_rangeslider_visible=False,
	#     yaxis=dict(
	#         title='Price',
	#         titlefont=dict(color='#1f77b4'),
	#     ),
	#     yaxis2=dict(
	#         title='Momentum',
	#         titlefont=dict(color='#ff7f0e'),
	#         anchor='free',
	#         overlaying='y',
	#         side='left',
	#         position=0.15
	#     ),
	#     yaxis3=dict(
	#         title='RSI',
	#         titlefont=dict(color='#d62728'),
	#         anchor='x',
	#         overlaying='y',
	#         side='right'
	#     ),
	# )

	# fig.show()


# Test

In [ ]:
# from datetime import datetime, timedelta
# import schedule
# import time

# symbol = 'ETHUSDT'
# from_date = (datetime.now() - timedelta(days=60)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
# to_date = (datetime.now() + timedelta(days=0)).strftime('%Y-%m-%d')
# timeframe = '1d'

# data = CommonBinance.CommonBinance.loaddataBinance_FromTo(symbol, from_date, to_date, timeframe)
# data

# Auto detect Entry Point

In [ ]:
from datetime import datetime, timedelta
import schedule
import time

def schedule_detectSignal(timeframe):
	symbol = 'ETHUSDT'
	from_date = (datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
	to_date = (datetime.now() + timedelta(days=0)).strftime('%Y-%m-%d')
	detectSignal(symbol, from_date, to_date, timeframe)

# Danh sách các phút cụ thể bạn muốn hàm được chạy
run_minutes = list(range(0, 60, 1)) # Tu 0 den 59
while True:
	current_time = datetime.now()
	current_minute = current_time.minute
	current_second = current_time.second
	# Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
	if current_minute in run_minutes: # Nếu có, gọi hàm detectSignal
		if current_second == 3: # Chỉ chạy hàm khi giây là 0

			print('Chay')
			print(datetime.now())

			# Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
			last_run = getattr(schedule_detectSignal, 'last_run', None) # Lấy phút cuối cùng mà hàm đã chạy
			# Nếu chưa chạy trong phút hiện tại, gọi hàm và lưu lại phút hiện tại
			if last_run is None or last_run != current_minute:
				schedule_detectSignal('1m')
				# Lưu lại phút cuối cùng mà hàm đã chạy
				setattr(schedule_detectSignal, 'last_run', current_minute)

			print('Chay xong')
			print(datetime.now())

	# Nghỉ 1 giây trước khi kiểm tra lại
	time.sleep(1)